# Fixed-Width Mixed-Prompt Addition: Balanced Seed Comparison

This notebook compares the balanced visible-length fixed-width mixed-prompt addition runs from
`artifacts/runs/addition_fixedwidth_mixed_20260424_180824`.

It includes:
- seed sanity summary,
- fixed-binary baseline heatmaps,
- original/random-composition heatmaps,
- direct comparison between fixed-binary and random `with_carry_filtered`,
- boundary-carry vs no-boundary-carry composed-eval diagnostics.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_ID = 'addition_fixedwidth_mixed_20260424_180824'
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'artifacts/runs' / RUN_ID / 'seed/seed_fit_results.json').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(f'Could not find artifacts/runs/{RUN_ID} from {Path.cwd()}')
RUN_ROOT = ROOT / 'artifacts/runs' / RUN_ID
ANALYSIS_DIR = ROOT / 'artifacts/analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 220,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

def load_json(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

seed = load_json(RUN_ROOT / 'seed/seed_fit_results.json')
paths = {
    ('fixed_binary', 'short_only'): RUN_ROOT / 'fullpack/short_only/self_improvement_results.json',
    ('fixed_binary', 'direct'): RUN_ROOT / 'fullpack/direct/self_improvement_results.json',
    ('fixed_binary', 'with_carry'): RUN_ROOT / 'fullpack/with_carry/self_improvement_results.json',
    ('fixed_binary', 'with_carry_filtered'): RUN_ROOT / 'fullpack/with_carry_filtered/self_improvement_results.json',
    ('random', 'direct'): RUN_ROOT / 'fullpack_original_composition/direct/self_improvement_results.json',
    ('random', 'with_carry'): RUN_ROOT / 'fullpack_original_composition/with_carry/self_improvement_results.json',
    ('random', 'with_carry_filtered'): RUN_ROOT / 'fullpack_original_composition/with_carry_filtered/self_improvement_results.json',
}
results = {key: load_json(path) for key, path in paths.items()}
print('Loaded', len(results), 'runs from', RUN_ROOT)

In [ ]:
seed_summary = {
    'validation_min_per_width': seed.get('validation_min_per_size_accuracy'),
    'test_min_per_width': seed.get('test_min_per_size_accuracy'),
    'meets_threshold': seed.get('meets_threshold'),
    'model_path': str(RUN_ROOT / 'seed/model'),
}
seed_summary

In [ ]:
def final_row(records):
    return records[-1]

def best_expanded_row(records):
    return max(records, key=lambda row: -1 if row.get('expanded_eval_accuracy') is None else row['expanded_eval_accuracy'])

summary_rows = []
for (composition, baseline), records in results.items():
    last = final_row(records)
    best = best_expanded_row(records)
    summary_rows.append({
        'composition': composition,
        'baseline': baseline,
        'final_eval': last.get('eval_accuracy'),
        'final_expanded_eval': last.get('expanded_eval_accuracy'),
        'final_frontier_train': last.get('frontier_train_accuracy'),
        'final_seed_eval': last.get('seed_eval_accuracy'),
        'final_composed_eval': last.get('stitched_all_composed_accuracy'),
        'final_no_boundary_eval': last.get('retained_no_boundary_carry_accuracy'),
        'final_boundary_eval': last.get('filtered_out_boundary_carry_accuracy'),
        'best_expanded_round': best.get('round'),
        'best_expanded_eval': best.get('expanded_eval_accuracy'),
    })
summary_df = pd.DataFrame(summary_rows).sort_values(['composition', 'baseline'])
summary_df.style.format(precision=4)

In [ ]:
def heatmap_matrix(records):
    rounds = [int(row['round']) for row in records]
    digits = sorted({int(d) for row in records for d in row.get('per_digit_accuracy', {}).keys()})
    mat = np.full((len(digits), len(rounds)), np.nan)
    for j, row in enumerate(records):
        per = row.get('per_digit_accuracy', {})
        for i, digit in enumerate(digits):
            value = per.get(str(digit))
            if value is not None:
                mat[i, j] = float(value)
    return digits, rounds, mat

def plot_heatmap(ax, records, title, annotate='sparse', cmap='viridis'):
    digits, rounds, mat = heatmap_matrix(records)
    im = ax.imshow(mat, aspect='auto', origin='upper', vmin=0, vmax=1, cmap=cmap)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Self-improvement round')
    ax.set_ylabel('Internal operand width')
    ax.set_xticks(range(len(rounds)))
    ax.set_xticklabels(rounds)
    stride = 1 if len(digits) <= 12 else 2
    y_positions = list(range(0, len(digits), stride))
    if (len(digits) - 1) not in y_positions:
        y_positions.append(len(digits) - 1)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([digits[i] for i in y_positions])
    if annotate != 'none':
        for i, digit in enumerate(digits):
            for j, rnd in enumerate(rounds):
                should_annotate = annotate == 'all' or j in {0, len(rounds)-1} or i in y_positions
                if should_annotate and not np.isnan(mat[i, j]):
                    color = 'white' if mat[i, j] < 0.45 else 'black'
                    ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=7, color=color)
    return im

def save_fig(fig, name):
    pdf = ANALYSIS_DIR / f'{name}.pdf'
    png = ANALYSIS_DIR / f'{name}.png'
    fig.savefig(pdf, bbox_inches='tight')
    fig.savefig(png, bbox_inches='tight')
    print('saved', pdf.relative_to(ROOT), 'and', png.relative_to(ROOT))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), constrained_layout=True)
fixed_order = ['short_only', 'direct', 'with_carry', 'with_carry_filtered']
for ax, baseline in zip(axes.ravel(), fixed_order):
    im = plot_heatmap(ax, results[('fixed_binary', baseline)], baseline.replace('_', ' ').title(), annotate='sparse')
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label='Held-out accuracy')
fig.suptitle('Fixed-binary composition path', fontsize=17, fontweight='bold')
save_fig(fig, 'addition_fixedwidth_mixed_fixed_binary_heatmaps')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
random_order = ['direct', 'with_carry', 'with_carry_filtered']
for ax, baseline in zip(axes, random_order):
    im = plot_heatmap(ax, results[('random', baseline)], baseline.replace('_', ' ').title(), annotate='sparse')
fig.colorbar(im, ax=axes.tolist(), shrink=0.85, label='Held-out accuracy')
fig.suptitle('Original/random composition path', fontsize=17, fontweight='bold')
save_fig(fig, 'addition_fixedwidth_mixed_random_composition_heatmaps')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
fb_records = results[('fixed_binary', 'with_carry_filtered')]
rand_records = results[('random', 'with_carry_filtered')]
im = plot_heatmap(axes[0], fb_records, 'Fixed Binary WCF', annotate='sparse')
plot_heatmap(axes[1], rand_records, 'Random WCF', annotate='sparse')

fb_digits, fb_rounds, fb_mat = heatmap_matrix(fb_records)
rd_digits, rd_rounds, rd_mat = heatmap_matrix(rand_records)
assert fb_digits == rd_digits and fb_rounds == rd_rounds
diff = fb_mat - rd_mat
im_diff = axes[2].imshow(diff, aspect='auto', origin='upper', vmin=-0.25, vmax=0.25, cmap='coolwarm')
axes[2].set_title('Fixed - Random', fontweight='bold')
axes[2].set_xlabel('Self-improvement round')
axes[2].set_ylabel('Internal operand width')
axes[2].set_xticks(range(len(fb_rounds)))
axes[2].set_xticklabels(fb_rounds)
y_positions = list(range(0, len(fb_digits), 2))
if len(fb_digits)-1 not in y_positions:
    y_positions.append(len(fb_digits)-1)
axes[2].set_yticks(y_positions)
axes[2].set_yticklabels([fb_digits[i] for i in y_positions])
for i in y_positions:
    for j in range(len(fb_rounds)):
        axes[2].text(j, i, f'{diff[i,j]:+.2f}', ha='center', va='center', fontsize=7, color='black')
fig.colorbar(im, ax=axes[:2].tolist(), shrink=0.85, label='Held-out accuracy')
fig.colorbar(im_diff, ax=axes[2], shrink=0.85, label='Accuracy diff')
fig.suptitle('with_carry_filtered: fixed binary vs original/random path', fontsize=16, fontweight='bold')
save_fig(fig, 'addition_fixedwidth_mixed_wcf_fixed_vs_random')
plt.show()

In [ ]:
def curve(records, key):
    xs = [r['round'] for r in records]
    ys = [np.nan if r.get(key) is None else float(r[key]) for r in records]
    return xs, ys

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for baseline in fixed_order:
    xs, ys = curve(results[('fixed_binary', baseline)], 'expanded_eval_accuracy')
    axes[0].plot(xs, ys, marker='o', label=baseline)
axes[0].set_title('Fixed-binary expanded held-out accuracy')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Expanded eval accuracy')
axes[0].set_ylim(0, 1.02)
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=9)

for composition in ['fixed_binary', 'random']:
    records = results[(composition, 'with_carry_filtered')]
    for key, label, style in [
        ('expanded_eval_accuracy', 'expanded eval', '-'),
        ('frontier_train_accuracy', 'frontier train', '--'),
        ('seed_eval_accuracy', 'seed eval', ':'),
    ]:
        xs, ys = curve(records, key)
        axes[1].plot(xs, ys, marker='o', linestyle=style, label=f'{composition}: {label}')
axes[1].set_title('WCF stability metrics')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.02)
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)
save_fig(fig, 'addition_fixedwidth_mixed_metric_curves')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for composition in ['fixed_binary', 'random']:
    records = results[(composition, 'with_carry_filtered')]
    for key, label, style in [
        ('retained_no_boundary_carry_accuracy', 'no-boundary retained', '-'),
        ('filtered_out_boundary_carry_accuracy', 'boundary filtered-out', '--'),
        ('stitched_all_composed_accuracy', 'all composed eval', ':'),
    ]:
        xs, ys = curve(records, key)
        ax.plot(xs, ys, marker='o', linestyle=style, label=f'{composition}: {label}')
ax.set_title('with_carry_filtered composed-eval slices')
ax.set_xlabel('Round')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncol=2)
save_fig(fig, 'addition_fixedwidth_mixed_wcf_boundary_slice_curves')
plt.show()

In [ ]:
# Compact final-round comparison table for copy/paste.
final_cols = [
    'composition', 'baseline', 'final_expanded_eval', 'final_frontier_train',
    'final_seed_eval', 'final_composed_eval', 'final_no_boundary_eval', 'final_boundary_eval'
]
summary_df[final_cols].sort_values('final_expanded_eval', ascending=False).style.format(precision=4)

## Quick read

The balanced seed succeeds, and `with_carry_filtered` is the strongest condition.
Fixed-binary WCF is slightly better on final natural expanded eval, while random WCF has a stronger final boundary-carry filtered-out composed-eval slice in this run.